# 3-1절 연습 문제 풀이

이 노트북은 3-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch03/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 3-1

다음 조건의 다층 퍼셉트론에 포함된 가중치 파라미터, 편향 파라미터, 전체 파라미터의 수를 각각 계산해 보자.

세 개의 값을 입력받아 두 개의 값을 출력하는 다층 퍼셉트론

뉴런의 수가 각각 네 개, 두 개인 은닉층 두 개를 포함

In [ ]:
# 3 -> 4 -> 2 -> 2 구조의 다층 퍼셉트론
layers = [(3, 4), (4, 2), (2, 2)]    # (입력 크기, 출력 크기)
total_w = total_b = 0
for i, (fan_in, fan_out) in enumerate(layers, start=1):
    w, b = fan_in * fan_out, fan_out
    total_w += w; total_b += b
    print(f'{i}번째 계층 ({fan_in} -> {fan_out}): 가중치 {w:2d}개, 편향 {b}개')
print(f'\n가중치 {total_w}개 + 편향 {total_b}개 = 전체 {total_w + total_b}개')

# 파이토치 모델로 검증
model = nn.Sequential(nn.Linear(3, 4), nn.Sigmoid(),
                      nn.Linear(4, 2), nn.Sigmoid(),
                      nn.Linear(2, 2))
print(f'검증: {sum(p.numel() for p in model.parameters())}개')

각 선형 계층의 가중치 수는 (입력 크기 × 출력 크기), 편향 수는 출력 크기다. 활성화 계층에는 파라미터가 없다. 따라서 가중치 12+8+4=24개, 편향 4+2+2=8개, 전체 **32개**다.

## 연습 3-2

[그림 3-6]을 참고해 다음 조건의 다층 퍼셉트론의 구조를 종이에 그려 보자. 층 사이의 연결선은 자세히 그리지 않아도 된다.

3x2 크기의 이미지를 입력받고, 2x3 크기의 이미지를 출력하는 다층 퍼셉트론(이미지의 각 픽셀은 0과 1 사이의 값 하나로 표현된다.)

뉴런의 수가 각각 열 개, 여덟 개인 은닉층 두 개를 포함

힌트: 입력과 출력 이미지의 형태는 무시하고 픽셀 수만 생각한다.

### 풀이

입력 이미지는 3×2 = **6픽셀**, 출력 이미지는 2×3 = **6픽셀**이므로 형태와 무관하게 입력 6개, 출력 6개로 생각하면 된다. 은닉층은 뉴런 10개와 8개다.

```
입력층(6)  →  은닉층1(10)  →  은닉층2(8)  →  출력층(6)
 ○ ○ ○         ○ ○ ○ ○ ○      ○ ○ ○ ○        ○ ○ ○
 ○ ○ ○         ○ ○ ○ ○ ○      ○ ○ ○ ○        ○ ○ ○
              (10개)          (8개)
```

각 층은 앞 층의 모든 뉴런과 연결된다(완전 연결). 그림에서 연결선은 생략해도 되지만, 파라미터 수는 아래 코드로 확인할 수 있다.

In [ ]:
model = nn.Sequential(
    nn.Linear(6, 10), nn.Sigmoid(),     # 3x2 이미지 = 6픽셀 입력
    nn.Linear(10, 8), nn.Sigmoid(),
    nn.Linear(8, 6), nn.Sigmoid(),      # 2x3 이미지 = 6픽셀 출력
)
for name, p in model.named_parameters():
    print(f'{name:14s} {tuple(p.shape)}')
print(f'전체 파라미터: {sum(p.numel() for p in model.parameters())}개')

## 연습 3-3

다음 그림과 같은 다층 퍼셉트론에서 입력 (x1, x2)가 (-1, -1), (-1, 1), (1, -1), (1, 1)일 때의 출력값을 각각 계산해 보자. 단, 모든 뉴런의 편향은 1로, 가중치는 각 뉴런에 표시된 숫자를 사용하며 계산 편의를 위해 활성화 함수는 입력한 값을 그대로 출력하는 항등 함수(σx=x)로 가정한다.

그림 3-7 모든 뉴런의 편향은 1이며, 가중치는 뉴런 내부에 표시한 다층 퍼셉트론

In [ ]:
# 활성화 함수가 항등 함수이고 모든 편향이 1인 경우의 출력 계산
# 각 뉴런: 출력 = (입력 x 가중치)의 합 + 1
def forward(x1, x2, w_hidden, w_out):
    # w_hidden: [[뉴런1의 w1, w2], [뉴런2의 w1, w2]]
    h = [w[0] * x1 + w[1] * x2 + 1 for w in w_hidden]
    return sum(w * hi for w, hi in zip(w_out, h)) + 1

# 아래 값은 예시다. 책 그림에 표시된 숫자로 바꿔 넣으면 그대로 계산된다.
W_HIDDEN = [[1, 2], [3, -1]]      # 은닉층 뉴런 두 개
W_OUT = [1, 1]                    # 출력 뉴런

print(f'{"입력":>10} {"은닉층 출력":>18} {"최종 출력":>10}')
for x1, x2 in [(-1, -1), (-1, 1), (1, -1), (1, 1)]:
    h = [w[0] * x1 + w[1] * x2 + 1 for w in W_HIDDEN]
    print(f'{str((x1, x2)):>10} {str([round(v, 1) for v in h]):>18} '
          f'{forward(x1, x2, W_HIDDEN, W_OUT):>10.1f}')

활성화 함수가 입력을 그대로 통과시키는 항등 함수이므로 각 뉴런의 출력은 `가중합 + 편향(1)`이다.

> 가중치 값은 책 그림에 표시된 숫자로 바꿔 넣어야 한다. 위 코드는 계산 절차를 보여주는 예시이며, `W_HIDDEN`과 `W_OUT`만 교체하면 어떤 가중치에도 적용된다.

실행 결과를 정리하면 최종 출력은 `4·x1 + x2 + 3`이라는 **하나의 일차식**과 정확히 일치한다. 은닉층을 거쳤는데도 결과가 단순한 일차식이 되는 이유는 활성화 함수가 선형(항등)이기 때문이다.

**활성화 함수가 선형이면 층을 아무리 쌓아도 전체가 하나의 선형 변환으로 축약된다.** 비선형 활성화 함수가 반드시 필요한 이유가 여기에 있다.